# Service Notebook: Cleanup External Storage Paths & Drop Tables in ADB
**Framework:** Metadata-Driven Pipeline Framework  
**Purpose:** Environment reset / maintenance utility. Drops Unity Catalog tables/volumes and recursively deletes external ADLS Gen2 storage paths (Landing and Bronze).

In [ ]:
# 1. Define Input Widgets
import json

dbutils.widgets.text("source_table_name", "ALL", "Table Name (e.g. Policies or ALL)")
dbutils.widgets.text("target_catalog", "bronze_catalog", "Target Catalog")
dbutils.widgets.text("target_schema", "raw_data", "Target Schema")
dbutils.widgets.text("volume_catalog", "landing_catalog", "Volume Catalog")
dbutils.widgets.text("volume_schema", "source_sql", "Volume Schema")
dbutils.widgets.text("landing_container", "landing", "Landing Container")
dbutils.widgets.text("bronze_container", "bronze", "Bronze Container")
dbutils.widgets.text("raw_container", "bronze", "Raw Container (Fallback)")
dbutils.widgets.text("storage_account_name", "azureprojectdevadls01", "Storage Account Name")
dbutils.widgets.text("folder_prefix", "home/datalake/claims", "Folder Prefix")
dbutils.widgets.text("drop_uc_table", "true", "Drop Unity Catalog Table (true/false)")
dbutils.widgets.text("delete_landing_path", "true", "Delete Landing Path (true/false)")
dbutils.widgets.text("delete_bronze_path", "true", "Delete Bronze Path (true/false)")
dbutils.widgets.text("delete_raw_path", "true", "Delete Raw Path (true/false)")
dbutils.widgets.text("drop_uc_volume", "false", "Drop UC Volume (true/false)")

source_table_name    = dbutils.widgets.get("source_table_name").strip()
target_catalog       = dbutils.widgets.get("target_catalog").strip()
target_schema        = dbutils.widgets.get("target_schema").strip()
volume_catalog       = dbutils.widgets.get("volume_catalog").strip()
volume_schema        = dbutils.widgets.get("volume_schema").strip()
landing_container    = dbutils.widgets.get("landing_container").strip()

try:
    bronze_container = dbutils.widgets.get("bronze_container").strip()
except Exception:
    try:
        bronze_container = dbutils.widgets.get("raw_container").strip()
    except Exception:
        bronze_container = "bronze"

raw_container        = bronze_container
storage_account_name = dbutils.widgets.get("storage_account_name").strip()
folder_prefix        = dbutils.widgets.get("folder_prefix").strip().strip("/")
drop_uc_table        = dbutils.widgets.get("drop_uc_table").strip().lower() in ["true", "y", "1", "yes"]
delete_landing_path  = dbutils.widgets.get("delete_landing_path").strip().lower() in ["true", "y", "1", "yes"]
delete_bronze_path   = dbutils.widgets.get("delete_bronze_path").strip().lower() in ["true", "y", "1", "yes"]
delete_raw_path      = dbutils.widgets.get("delete_raw_path").strip().lower() in ["true", "y", "1", "yes"]
drop_uc_volume       = dbutils.widgets.get("drop_uc_volume").strip().lower() in ["true", "y", "1", "yes"]

print(f"Cleanup Target: {source_table_name}")
print(f"Catalog / Schema: `{target_catalog}`.`{target_schema}`")
print(f"Landing Container: {landing_container}")
print(f"Bronze Container: {bronze_container}")
print(f"Drop UC Table: {drop_uc_table}")
print(f"Delete Landing Storage: {delete_landing_path}")
print(f"Delete Bronze Storage: {delete_bronze_path}")
print(f"Drop UC Volume: {drop_uc_volume}")

In [ ]:
# 2. Identify Target Tables to Clean
# Standard active tables in Medallion framework (Landing & Bronze)
all_standard_tables = ["Policies", "Customers", "CoverageTypes", "Claims", "Agents"]

if source_table_name.upper() == "ALL" or not source_table_name:
    try:
        existing_tables = [t.name for t in spark.catalog.listTables(f"{target_catalog}.{target_schema}")]
        # Union discovered catalog tables with standard tables to ensure storage paths are cleaned even if table was already dropped
        tables_to_clean = sorted(list(set(existing_tables + all_standard_tables)))
        print(f"Discovered & standard tables: {tables_to_clean}")
    except Exception as e_list:
        print(f"Could not list tables in `{target_catalog}`.`{target_schema}`: {str(e_list)}")
        tables_to_clean = all_standard_tables
else:
    tables_to_clean = [t.strip() for t in source_table_name.split(",") if t.strip()]

print(f"Tables to process ({len(tables_to_clean)}): {tables_to_clean}")

In [ ]:
# 3. Execute Cleanup Operations (Drop Table + Delete Storage Paths + Drop Volume)
results = []

for tbl in tables_to_clean:
    tbl_result = {"table": tbl, "actions": []}
    print(f"\n--- Processing Table: {tbl} ---")
    
    # 3a. Drop Unity Catalog Bronze Table
    if drop_uc_table:
        full_tbl = f"`{target_catalog}`.`{target_schema}`.`{tbl}`"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_tbl}")
            print(f"[ADB] Dropped table {full_tbl}")
            tbl_result["actions"].append(f"Dropped table {full_tbl}")
        except Exception as e_drop:
            print(f"[ADB WARNING] Failed to drop table {full_tbl}: {str(e_drop)}")
            tbl_result["actions"].append(f"Drop table error: {str(e_drop)}")
            
    # 3b. Delete Landing Storage Path (CSVs: abfss://landing@.../home/datalake/claims/{tbl})
    if delete_landing_path:
        landing_path = f"abfss://{landing_container}@{storage_account_name}.dfs.core.windows.net/{folder_prefix}/{tbl}"
        try:
            dbutils.fs.rm(landing_path, recurse=True)
            print(f"[ADLS] Deleted Landing path: {landing_path}")
            tbl_result["actions"].append(f"Deleted Landing storage: {landing_path}")
        except Exception as e_land:
            print(f"[ADLS NOTE] Landing path {landing_path} not found or could not be removed: {str(e_land)}")
            tbl_result["actions"].append(f"Landing storage note: {str(e_land)}")
            
    # 3c. Delete Bronze Storage Path (Delta & Parquet: abfss://bronze@.../home/datalake/claims/{tbl})
    if delete_bronze_path or delete_raw_path:
        bronze_path = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net/{folder_prefix}/{tbl}"
        try:
            dbutils.fs.rm(bronze_path, recurse=True)
            print(f"[ADLS] Deleted Bronze path: {bronze_path}")
            tbl_result["actions"].append(f"Deleted Bronze storage: {bronze_path}")
        except Exception as e_brz:
            print(f"[ADLS NOTE] Bronze path {bronze_path} not found or could not be removed: {str(e_brz)}")
            tbl_result["actions"].append(f"Bronze storage note: {str(e_brz)}")
            
    # 3d. Optionally Drop Unity Catalog Landing Volume (vol_<table_name>)
    if drop_uc_volume:
        vol_name = f"vol_{tbl.lower()}"
        full_vol = f"`{volume_catalog}`.`{volume_schema}`.`{vol_name}`"
        try:
            spark.sql(f"DROP VOLUME IF EXISTS {full_vol}")
            print(f"[UC] Dropped volume {full_vol}")
            tbl_result["actions"].append(f"Dropped volume {full_vol}")
        except Exception as e_vol:
            print(f"[UC NOTE] Volume {full_vol} note: {str(e_vol)}")
            
    results.append(tbl_result)

print("\n--- Cleanup Operations Completed ---")

In [ ]:
# 4. Return Output JSON to ADF
output = {
    "status": "Success",
    "tables_processed": len(tables_to_clean),
    "table_names": tables_to_clean,
    "details": results
}

print(json.dumps(output, indent=2))
dbutils.notebook.exit(json.dumps(output))